In [29]:
# Cell 1 — imports and connection
import os
import pandas as pd
from dotenv import load_dotenv
import snowflake.connector
from collections import Counter
import json
 
load_dotenv()
 
conn = snowflake.connector.connect(
    user=os.getenv('SNOWFLAKE_USER'),
    password=os.getenv('SNOWFLAKE_PASSWORD'),
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
    database='ANALYTICS_PROD',
    schema='PUBLIC',
)
 
def run_query(sql: str) -> pd.DataFrame:
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [desc[0] for desc in cur.description]
    cur.close()
    return pd.DataFrame(rows, columns=cols)
 
print("Connected.")

Connected.


In [30]:
# ── Cell 2 — load mart ────────────────────────────────────────────────────────
df = run_query("SELECT * FROM ANALYTICS_PROD.PUBLIC.FCT_JOB_POSTINGS")
df.columns = [c.lower() for c in df.columns]

# Parse arrays
def parse_arr(val):
    if isinstance(val, list): return val
    if isinstance(val, str):
        try: return json.loads(val)
        except: return []
    return []

for col in ["tech_stack_required", "tech_stack_preferred", "paradigms_required", "paradigms_preferred"]:
    df[col] = df[col].apply(parse_arr)

# Parse dates
for col in ["date_posted", "ingested_at", "enriched_at"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Numeric
for col in ["final_salary_min", "final_salary_max", "years_required_min", "years_required_max", "confidence_score"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Booleans
for col in ["acknowledges_ai", "explicitly_encourages_applicants"]:
    df[col] = df[col].apply(lambda x: True if str(x).strip().upper() in ("TRUE", "1", "YES") else False)

# early_career_tier comes straight from the mart now — no notebook-side computation needed
EARLY_CAREER_ORDER = ["entry_or_junior", "mid"]

# Salary mid
salary_df = df.dropna(subset=["final_salary_min", "final_salary_max"]).copy()
salary_df["salary_mid"] = (salary_df["final_salary_min"] + salary_df["final_salary_max"]) / 2

print(f"Loaded {len(df)} postings")

Loaded 364 postings


In [31]:
# ── Cell 3 — top level snapshot ───────────────────────────────────────────────
print("=== TOP LEVEL ===")
print(f"Total postings:       {len(df)}")
print(f"Unique companies:     {df['company_name'].nunique()}")
print(f"Role types:           {df['ingestion_query'].nunique()}")
print(f"Sources:              {df['source'].nunique()}")
print(f"Salary disclosed:     {len(salary_df)} ({len(salary_df)/len(df):.0%})")
print(f"LLM enriched:         {df['role_archetype'].notna().sum()} ({df['role_archetype'].notna().mean():.0%})")
print(f"Date range:           {df['date_posted'].min().date()} → {df['date_posted'].max().date()}")
print(f"Last ingested:        {df['ingested_at'].max().date()}")
 

=== TOP LEVEL ===
Total postings:       364
Unique companies:     303
Role types:           4
Sources:              3
Salary disclosed:     217 (60%)
LLM enriched:         364 (100%)
Date range:           2026-05-20 → 2026-06-19
Last ingested:        2026-06-20


In [32]:
# ── Cell 4 — postings by role type ───────────────────────────────────────────
print("\n=== POSTINGS BY ROLE TYPE ===")
print(df["ingestion_query"].value_counts().to_string())


=== POSTINGS BY ROLE TYPE ===
ingestion_query
Data Analyst          182
Data Scientist         86
Data Engineer          60
Analytics Engineer     36


In [33]:
# ── Cell 5 — postings by source ──────────────────────────────────────────────
print("\n=== POSTINGS BY SOURCE ===")
print(df["source"].value_counts().to_string())
 
print("\n=== POSTINGS BY ROLE × SOURCE ===")
print(df.groupby(["ingestion_query", "source"]).size().unstack(fill_value=0).to_string())


=== POSTINGS BY SOURCE ===
source
jsearch       130
theirstack    125
builtin       109

=== POSTINGS BY ROLE × SOURCE ===
source              builtin  jsearch  theirstack
ingestion_query                                 
Analytics Engineer        3       24           9
Data Analyst             50       78          54
Data Engineer            22       10          28
Data Scientist           34       18          34


In [34]:
# ── Cell 6 — work model ───────────────────────────────────────────────────────
print("\n=== WORK MODEL (overall) ===")
print(df["work_model"].value_counts().to_string())
 
print("\n=== WORK MODEL by ROLE ===")
print(df.groupby(["ingestion_query", "work_model"]).size().unstack(fill_value=0).to_string())


=== WORK MODEL (overall) ===
work_model
onsite    213
remote    110
hybrid     41

=== WORK MODEL by ROLE ===
work_model          hybrid  onsite  remote
ingestion_query                           
Analytics Engineer       3      27       6
Data Analyst            17     121      44
Data Engineer           12      21      27
Data Scientist           9      44      33


In [35]:
# ── Cell 7 — early-career tier distribution ──────────────────────────────────
print("\n=== EARLY CAREER TIER (overall) ===")
print(df["early_career_tier"].value_counts().to_string())

print("\n=== EARLY CAREER TIER by ROLE ===")
print(df.groupby(["ingestion_query", "early_career_tier"]).size().unstack(fill_value=0).to_string())


=== EARLY CAREER TIER (overall) ===
early_career_tier
mid                173
entry_or_junior     49

=== EARLY CAREER TIER by ROLE ===
early_career_tier   entry_or_junior  mid
ingestion_query                         
Analytics Engineer                0   12
Data Analyst                     26   66
Data Engineer                    10   40
Data Scientist                   13   55


In [36]:
# ── Cell 8 — salary by role ───────────────────────────────────────────────────
print("\n=== SALARY by ROLE (median, where disclosed) ===")
sal_by_role = (
    salary_df.groupby("ingestion_query")["salary_mid"]
    .agg(["median", "count", "min", "max", "std"])
    .round(0)
)
sal_by_role.columns = ["median", "n", "min", "max", "std"]
print(sal_by_role.to_string())

print("\n=== SALARY by ROLE × EARLY CAREER TIER ===")
sal_tier = salary_df[salary_df["early_career_tier"].isin(EARLY_CAREER_ORDER)]
print(
    sal_tier.groupby(["ingestion_query", "early_career_tier"])["salary_mid"]
    .agg(["median", "count"])
    .round(0)
    .to_string()
)


=== SALARY by ROLE (median, where disclosed) ===
                      median    n       min        max       std
ingestion_query                                                 
Analytics Engineer  142064.0   25   55000.0   445000.0   75455.0
Data Analyst         93400.0  101    2080.0   167500.0   29166.0
Data Engineer       140000.0   31  102500.0  1100000.0  174571.0
Data Scientist      141000.0   60       2.0   450000.0   65710.0

=== SALARY by ROLE × EARLY CAREER TIER ===
                                        median  count
ingestion_query    early_career_tier                 
Analytics Engineer mid                133750.0     10
Data Analyst       entry_or_junior     92500.0     13
                   mid                100000.0     37
Data Engineer      entry_or_junior    120000.0      5
                   mid                142200.0     22
Data Scientist     entry_or_junior     94525.0      8
                   mid                143350.0     39


In [37]:
# ── Cell 9 — AI blindspot ─────────────────────────────────────────────────────
print("\n=== AI ACKNOWLEDGMENT (overall) ===")
print(f"Acknowledges AI: {df['acknowledges_ai'].sum()} of {len(df)} ({df['acknowledges_ai'].mean():.0%})")
 
print("\n=== AI ACKNOWLEDGMENT by ROLE ===")
ai = (
    df.groupby("ingestion_query")["acknowledges_ai"]
    .agg(["sum", "count", "mean"])
    .round(3)
)
ai.columns = ["yes", "total", "rate"]
print(ai.to_string())


=== AI ACKNOWLEDGMENT (overall) ===
Acknowledges AI: 174 of 364 (48%)

=== AI ACKNOWLEDGMENT by ROLE ===
                    yes  total   rate
ingestion_query                      
Analytics Engineer   19     36  0.528
Data Analyst         48    182  0.264
Data Engineer        31     60  0.517
Data Scientist       76     86  0.884


In [38]:
# ── Cell 10 — title vs archetype confusion ────────────────────────────────────
print("\n=== TITLE vs LLM ARCHETYPE (confusion matrix, counts) ===")
matrix_df = df.dropna(subset=["role_archetype"]).copy()
pivot = (
    matrix_df.groupby(["ingestion_query", "role_archetype"])
    .size()
    .unstack(fill_value=0)
)
print(pivot.to_string())
 
print("\n=== TITLE vs LLM ARCHETYPE (row %, agreement on diagonal) ===")
print(pivot.div(pivot.sum(axis=1), axis=0).round(2).to_string())
 
# Agreement rate
def queries_match(row):
    q = row["ingestion_query"].lower().replace(" ", "_").replace("-", "_")
    a = row["role_archetype"].lower()
    return bool(set(q.split("_")) & set(a.split("_")))
 
agree_n = matrix_df.apply(queries_match, axis=1).sum()
print(f"\nAgreement: {agree_n} of {len(matrix_df)} ({agree_n/len(matrix_df):.0%})")


=== TITLE vs LLM ARCHETYPE (confusion matrix, counts) ===
role_archetype      analytics_engineer  data_analyst  data_engineer  data_scientist  hybrid
ingestion_query                                                                            
Analytics Engineer                  16             1             16               1       2
Data Analyst                         1           164              2               6       9
Data Engineer                        0             2             55               1       2
Data Scientist                       0             0              0              85       1

=== TITLE vs LLM ARCHETYPE (row %, agreement on diagonal) ===
role_archetype      analytics_engineer  data_analyst  data_engineer  data_scientist  hybrid
ingestion_query                                                                            
Analytics Engineer                0.44          0.03           0.44            0.03    0.06
Data Analyst                      0.01          0.

In [39]:
# ── Cell 11 — top skills overall and by role ──────────────────────────────────
print("\n=== TOP 20 REQUIRED SKILLS (overall) ===")
all_req = [t for row in df["tech_stack_required"] if isinstance(row, list) for t in row if isinstance(t, str)]
print(pd.Series(Counter(all_req)).sort_values(ascending=False).head(20).to_string())
 
print("\n=== TOP 10 REQUIRED SKILLS by ROLE ===")
for query in df["ingestion_query"].unique():
    subset = df[df["ingestion_query"] == query]
    tools = [t for row in subset["tech_stack_required"] if isinstance(row, list) for t in row if isinstance(t, str)]
    top = pd.Series(Counter(tools)).sort_values(ascending=False).head(10)
    print(f"\n--- {query} ---")
    print(top.to_string())


=== TOP 20 REQUIRED SKILLS (overall) ===
sql             228
python          180
excel            75
r                51
power bi         43
tableau          42
snowflake        36
databricks       26
dbt              24
airflow          23
aws              22
bigquery         18
pyspark          17
looker           17
spark            16
pandas           16
git              15
powerpoint       14
github           14
scikit-learn     13

=== TOP 10 REQUIRED SKILLS by ROLE ===

--- Data Scientist ---
python          72
sql             58
r               22
pandas          10
scikit-learn    10
snowflake        9
pytorch          7
tensorflow       7
git              6
databricks       5

--- Data Analyst ---
sql           104
excel          69
python         44
power bi       31
tableau        31
r              26
looker         15
powerpoint     13
word           11
snowflake       9

--- Analytics Engineer ---
sql           28
python        22
airflow        9
snowflake      8
aws   

In [40]:
# ── Cell 12 — top paradigms by role ──────────────────────────────────────────
print("\n=== TOP 10 PARADIGMS by ROLE ===")
for query in df["ingestion_query"].unique():
    subset = df[df["ingestion_query"] == query]
    paras = [
        t
        for req, pref in zip(subset["paradigms_required"], subset["paradigms_preferred"])
        for row in [req, pref] if isinstance(row, list)
        for t in row if isinstance(t, str)
    ]
    top = pd.Series(Counter(paras)).sort_values(ascending=False).head(10)
    print(f"\n--- {query} ---")
    print(top.to_string())


=== TOP 10 PARADIGMS by ROLE ===

--- Data Scientist ---
machine learning        37
statistical analysis    26
data analysis           21
causal inference        14
predictive modeling     13
experimental design     12
nlp                     11
data quality            11
data modeling           10
data visualization       8

--- Data Analyst ---
data analysis           60
data visualization      46
data quality            42
data governance         30
statistical analysis    26
data validation         22
data modeling           21
data management         15
predictive modeling     11
data cleaning           10

--- Analytics Engineer ---
data modeling       22
etl design          15
data quality        15
data governance     13
data warehousing     9
ci/cd                8
responsible ai       2
data engineering     2
devops               2
data integration     2

--- Data Engineer ---
etl design                32
data modeling             29
data quality              26
data governa

In [41]:
# ── Cell 13 — experience requirements ────────────────────────────────────────
print("\n=== YEARS REQUIRED by ROLE (median, where specified) ===")
yrs = df.dropna(subset=["years_required_min"])
print(
    yrs.groupby("ingestion_query")["years_required_min"]
    .agg(["median", "count"])
    .round(1)
    .to_string()
)

print("\n=== YEARS REQUIRED by ROLE × EARLY CAREER TIER ===")
yrs_tier = yrs[yrs["early_career_tier"].isin(EARLY_CAREER_ORDER)]
print(
    yrs_tier.groupby(["ingestion_query", "early_career_tier"])["years_required_min"]
    .agg(["median", "count"])
    .round(1)
    .to_string()
)


=== YEARS REQUIRED by ROLE (median, where specified) ===
                    median  count
ingestion_query                  
Analytics Engineer     3.0     34
Data Analyst           2.0    154
Data Engineer          3.0     53
Data Scientist         3.0     79

=== YEARS REQUIRED by ROLE × EARLY CAREER TIER ===
                                      median  count
ingestion_query    early_career_tier               
Analytics Engineer mid                   3.0     12
Data Analyst       entry_or_junior       2.0     18
                   mid                   2.5     58
Data Engineer      entry_or_junior       0.5      8
                   mid                   3.0     37
Data Scientist     entry_or_junior       2.0     11
                   mid                   3.0     52


In [42]:
# ── Cell 14 — degree requirements ────────────────────────────────────────────
print("\n=== DEGREE REQUIREMENTS by ROLE ===")
deg = df.dropna(subset=["degree_requirement"])
print(
    deg.groupby(["ingestion_query", "degree_requirement"])
    .size()
    .unstack(fill_value=0)
    .to_string()
)


=== DEGREE REQUIREMENTS by ROLE ===
degree_requirement  bachelors  equivalent_ok  masters  none
ingestion_query                                            
Analytics Engineer         14              3        1    18
Data Analyst              101             14        9    58
Data Engineer              25              2        1    32
Data Scientist             30              4       21    31


In [44]:
# ── Cell 15 — encourages applicants ──────────────────────────────────────────
print("\n=== ENCOURAGES APPLICANTS (overall) ===")
print(f"Yes: {df['explicitly_encourages_applicants'].sum()} of {len(df)} ({df['explicitly_encourages_applicants'].mean():.1%})")

print("\n=== ENCOURAGES APPLICANTS by ROLE ===")
enc = (
    df.groupby("ingestion_query")["explicitly_encourages_applicants"]
    .agg(["sum", "count", "mean"])
    .round(3)
)
enc.columns = ["yes", "total", "rate"]
print(enc.to_string())


=== ENCOURAGES APPLICANTS (overall) ===
Yes: 42 of 364 (11.5%)

=== ENCOURAGES APPLICANTS by ROLE ===
                    yes  total   rate
ingestion_query                      
Analytics Engineer    3     36  0.083
Data Analyst         18    182  0.099
Data Engineer         7     60  0.117
Data Scientist       14     86  0.163


In [45]:
# ── listed vs. inferred seniority mismatch ────────────────────────
print("\n=== LISTED vs INFERRED SENIORITY (counts) ===")
mismatch_df = df.dropna(subset=["listed_seniority", "inferred_seniority"]).copy()
pivot = (
    mismatch_df.groupby(["listed_seniority", "inferred_seniority"])
    .size()
    .unstack(fill_value=0)
)
print(pivot.to_string())

print("\n=== LISTED vs INFERRED SENIORITY (row %, agreement on diagonal) ===")
print(pivot.div(pivot.sum(axis=1), axis=0).round(2).to_string())

print(f"\nTotal rows with both fields populated: {len(mismatch_df)}")

print("\n=== YEARS REQUIRED by LISTED SENIORITY (overlap check) ===")
yrs_listed = df.dropna(subset=["years_required_min", "listed_seniority"])
print(
    yrs_listed.groupby("listed_seniority")["years_required_min"]
    .agg(["median", "min", "max", "count"])
    .round(1)
    .to_string()
)


=== LISTED vs INFERRED SENIORITY (counts) ===
inferred_seniority  entry  junior  mid  senior
listed_seniority                              
entry_level             9       0    0       0
junior                  8      31    1       0
mid_level              14      39   94      26

=== LISTED vs INFERRED SENIORITY (row %, agreement on diagonal) ===
inferred_seniority  entry  junior   mid  senior
listed_seniority                               
entry_level          1.00    0.00  0.00    0.00
junior               0.20    0.78  0.02    0.00
mid_level            0.08    0.23  0.54    0.15

Total rows with both fields populated: 222

=== YEARS REQUIRED by LISTED SENIORITY (overlap check) ===
                  median  min   max  count
listed_seniority                          
entry_level          0.0  0.0   0.0      2
junior               2.0  0.0   3.0     35
mid_level            3.0  0.0  10.0    159
